
# Hu?n luy?n Q-MARL tr?n MATE: ?n ??nh actor/critic v? theo d?i balanced score

Notebook n?y c?p nh?t theo v?ng s?a m?i c?a Q-MARL:
- t?ng entropy regularization ?? gi?m policy collapse
- t?ch `actor_lr` v? `critic_lr`
- gi?m `max_grad_norm`
- d?ng critic ?n ??nh h?n v? log th?m c?c ch? s? n?i b?
- ch?n checkpoint kh?ng ch? theo reward m? c?n theo balanced score


In [ ]:
import importlib
import q_marl.config as qmarl_config
import q_marl.agent as qmarl_agent
import q_marl.networks as qmarl_networks
importlib.reload(qmarl_config)
importlib.reload(qmarl_networks)
importlib.reload(qmarl_agent)

In [ ]:
from pathlib import Path
import json
import time

import gymnasium as gym
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mate
from mate.agents import GreedyTargetAgent
from mate.wrappers.discrete_action_spaces import DiscreteCamera
from q_marl.agent import QMARL
from q_marl.config import QMARLConfig


## C?u h?nh ?? xu?t

C?u h?nh n?y t?p trung x? l? hai v?n ?? ch?nh ?? th?y ? c?c run tr??c:
- entropy s?p v? 0 qu? s?m
- critic loss t?ng m?nh v? TD error kh?ng ?n ??nh

C?c thay ??i ch?nh:
- `entropy_coef = 0.02`
- `actor_lr = 0.003`
- `critic_lr = 0.0005`
- `max_grad_norm = 0.5`
- reward normalization b?t m?c ??nh
- critic d?ng `Huber loss`


In [ ]:

RUN_NAME = "qmarl_mate_4v4_9_stable_actor_critic"
CONFIG_NAME = "MATE-4v4-9.yaml"

CHECKPOINT_DIR = Path("checkpoints")
RESULTS_DIR = Path("results")
CHECKPOINT_PATH = CHECKPOINT_DIR / f"{RUN_NAME}.pt"
BEST_REWARD_CHECKPOINT_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_best_reward.pt"
BEST_BALANCED_CHECKPOINT_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_best_balanced.pt"
TRAIN_HISTORY_CSV = RESULTS_DIR / f"{RUN_NAME}_train_history.csv"
TRAIN_HISTORY_JSON = RESULTS_DIR / f"{RUN_NAME}_train_history.json"
EVAL_HISTORY_CSV = RESULTS_DIR / f"{RUN_NAME}_eval_history.csv"
CONFIG_SNAPSHOT_JSON = RESULTS_DIR / f"{RUN_NAME}_config.json"

TOTAL_ENV_STEPS = 100_000
TRAIN_CHUNK_STEPS = 10_000
EVAL_EVERY_STEPS = 10_000
EVAL_EPISODES = 5
EVAL_MAX_STEPS = 500

NUM_ENVS = 4
ROLLOUT_LENGTH = 128
N_EPOCHS = 2
NUM_MINI_BATCHES = 2
GRAPH_DEPTH = 3
EDGE_DIM = 10
ACTION_LEVELS = 5
ACTOR_LR = 0.003
CRITIC_LR = 0.0005
ENTROPY_COEF = 0.02
VALUE_COEF = 0.5
CRITIC_EXTRA_STEPS = 1
MAX_GRAD_NORM = 0.5
SEED = 42

BALANCED_REWARD_WEIGHT = 1.0
BALANCED_COVERAGE_WEIGHT = 400.0
BALANCED_TRANSPORT_WEIGHT = 400.0

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

config = QMARLConfig(
    env_config=CONFIG_NAME,
    num_envs=NUM_ENVS,
    rollout_length=ROLLOUT_LENGTH,
    n_epochs=N_EPOCHS,
    num_mini_batches=NUM_MINI_BATCHES,
    graph_depth=GRAPH_DEPTH,
    edge_dim=EDGE_DIM,
    action_levels=ACTION_LEVELS,
    actor_lr=ACTOR_LR,
    critic_lr=CRITIC_LR,
    entropy_coef=ENTROPY_COEF,
    value_coef=VALUE_COEF,
    critic_extra_steps=CRITIC_EXTRA_STEPS,
    max_grad_norm=MAX_GRAD_NORM,
    normalize_rewards=True,
    critic_loss="huber",
    device="cpu",
    render_mode="rgb_array",
    seed=SEED,
)

config_snapshot = {
    "run_name": RUN_NAME,
    "config_name": CONFIG_NAME,
    "total_env_steps": TOTAL_ENV_STEPS,
    "train_chunk_steps": TRAIN_CHUNK_STEPS,
    "eval_every_steps": EVAL_EVERY_STEPS,
    "eval_episodes": EVAL_EPISODES,
    "eval_max_steps": EVAL_MAX_STEPS,
    "num_envs": NUM_ENVS,
    "rollout_length": ROLLOUT_LENGTH,
    "n_epochs": N_EPOCHS,
    "num_mini_batches": NUM_MINI_BATCHES,
    "graph_depth": GRAPH_DEPTH,
    "edge_dim": EDGE_DIM,
    "action_levels": ACTION_LEVELS,
    "actor_lr": ACTOR_LR,
    "critic_lr": CRITIC_LR,
    "entropy_coef": ENTROPY_COEF,
    "value_coef": VALUE_COEF,
    "critic_extra_steps": CRITIC_EXTRA_STEPS,
    "max_grad_norm": MAX_GRAD_NORM,
    "seed": SEED,
}

CONFIG_SNAPSHOT_JSON.write_text(json.dumps(config_snapshot, indent=2, ensure_ascii=False), encoding="utf-8")
config_snapshot


In [ ]:

def make_camera_env(render_mode="rgb_array"):
    base_env = gym.make("MultiAgentTracking-v0", config=CONFIG_NAME, render_mode=render_mode)
    return mate.MultiCamera.make(base_env, target_agent=GreedyTargetAgent())


def discrete_action_to_continuous(env, action_indices, levels=ACTION_LEVELS):
    grid = DiscreteCamera.discrete_action_grid(levels=levels).astype(np.float32)
    action_high = np.asarray([env.unwrapped.camera_rotation_step, env.unwrapped.camera_zooming_step], dtype=np.float32)
    action_indices = np.asarray(action_indices, dtype=np.int64).reshape(env.unwrapped.num_cameras)
    return action_high * grid[action_indices]


def evaluate_current_agent(agent, episodes=EVAL_EPISODES, max_steps=EVAL_MAX_STEPS):
    env = make_camera_env()
    rows = []
    try:
        for episode_idx in range(1, episodes + 1):
            observation, _ = env.reset(seed=episode_idx)
            episode_reward = 0.0
            coverage_rate = float(getattr(env.unwrapped, "coverage_rate", np.nan))
            transport_rate = float(getattr(env.unwrapped, "mean_transport_rate", np.nan))
            delivered_cargoes = float(getattr(env.unwrapped, "num_delivered_cargoes", 0.0))

            for _ in range(1, max_steps + 1):
                action_idx = agent.predict_env(env, observation, deterministic=True)
                action = discrete_action_to_continuous(env, action_idx)
                observation, reward, terminated, truncated, info = env.step(action)
                episode_reward += float(np.mean(reward)) if not np.isscalar(reward) else float(reward)
                done = bool(terminated or truncated)
                info0 = info[0] if isinstance(info, list) and len(info) > 0 and isinstance(info[0], dict) else (info if isinstance(info, dict) else {})
                coverage_rate = float(info0.get("coverage_rate", getattr(env.unwrapped, "coverage_rate", np.nan)))
                transport_rate = float(info0.get("mean_transport_rate", getattr(env.unwrapped, "mean_transport_rate", np.nan)))
                delivered_cargoes = float(info0.get("num_delivered_cargoes", getattr(env.unwrapped, "num_delivered_cargoes", 0.0)))
                if done:
                    break

            rows.append({
                "episode": episode_idx,
                "episode_reward": episode_reward,
                "coverage_rate": coverage_rate,
                "transport_rate": transport_rate,
                "delivered_cargoes": delivered_cargoes,
            })
    finally:
        env.close()

    eval_df = pd.DataFrame(rows)
    return {
        "eval_episode_reward": float(eval_df["episode_reward"].mean()),
        "eval_episode_reward_std": float(eval_df["episode_reward"].std(ddof=0)),
        "eval_coverage_rate": float(eval_df["coverage_rate"].mean()),
        "eval_coverage_rate_std": float(eval_df["coverage_rate"].std(ddof=0)),
        "eval_transport_rate": float(eval_df["transport_rate"].mean()),
        "eval_transport_rate_std": float(eval_df["transport_rate"].std(ddof=0)),
        "eval_delivered_cargoes": float(eval_df["delivered_cargoes"].mean()),
        "eval_delivered_cargoes_std": float(eval_df["delivered_cargoes"].std(ddof=0)),
        "eval_episode_rows": rows,
    }


def compute_balanced_score(row):
    return (
        BALANCED_REWARD_WEIGHT * float(row["eval_episode_reward"]) +
        BALANCED_COVERAGE_WEIGHT * float(row["eval_coverage_rate"]) +
        BALANCED_TRANSPORT_WEIGHT * float(row["eval_transport_rate"])
    )


In [ ]:

agent = QMARL(config=config)
history = []
last_eval_step = 0
best_eval_reward = None
best_balanced_score = None
num_chunks = (TOTAL_ENV_STEPS + TRAIN_CHUNK_STEPS - 1) // TRAIN_CHUNK_STEPS

try:
    for chunk_idx in range(1, num_chunks + 1):
        start_time = time.time()
        target_steps = min(agent.total_env_steps + TRAIN_CHUNK_STEPS, TOTAL_ENV_STEPS)
        stats = agent.learn(total_env_steps=target_steps)
        elapsed = time.time() - start_time
        current_env_steps = int(stats["total_env_steps"])
        progress = 100.0 * current_env_steps / TOTAL_ENV_STEPS

        record = {
            "chunk": chunk_idx,
            "env_steps": current_env_steps,
            "agent_steps": int(stats["total_agent_steps"]),
            "actor_loss": float(stats["actor_loss"]),
            "actor_total_loss": float(stats["actor_total_loss"]),
            "critic_loss": float(stats["critic_loss"]),
            "entropy": float(stats["entropy"]),
            "mean_delta": float(stats["mean_delta"]),
            "baseline_mean": float(stats["baseline_mean"]),
            "q_mean": float(stats["q_mean"]),
            "completed_episodes": int(stats["completed_episodes"]),
            "chunk_time_sec": float(elapsed),
            "progress_percent": float(progress),
            "graph_depth": config.graph_depth,
            "edge_dim": config.edge_dim,
            "action_levels": config.action_levels,
            "did_evaluate": False,
            "is_best_eval_reward": False,
            "is_best_balanced": False,
        }

        checkpoint_step_path = CHECKPOINT_DIR / f"{RUN_NAME}_step_{current_env_steps}.pt"
        agent.save(checkpoint_step_path)
        agent.save(CHECKPOINT_PATH)
        record["checkpoint_path"] = str(checkpoint_step_path)

        should_eval = (current_env_steps - last_eval_step >= EVAL_EVERY_STEPS) or (current_env_steps >= TOTAL_ENV_STEPS)
        if should_eval:
            eval_stats = evaluate_current_agent(agent)
            record.update(eval_stats)
            record["balanced_score"] = compute_balanced_score(record)
            record["did_evaluate"] = True
            last_eval_step = current_env_steps

            if (best_eval_reward is None) or (record["eval_episode_reward"] > best_eval_reward):
                best_eval_reward = record["eval_episode_reward"]
                record["is_best_eval_reward"] = True
                agent.save(BEST_REWARD_CHECKPOINT_PATH)

            if (best_balanced_score is None) or (record["balanced_score"] > best_balanced_score):
                best_balanced_score = record["balanced_score"]
                record["is_best_balanced"] = True
                agent.save(BEST_BALANCED_CHECKPOINT_PATH)

        history.append(record)
        pd.DataFrame(history).to_csv(TRAIN_HISTORY_CSV, index=False)
        TRAIN_HISTORY_JSON.write_text(json.dumps(history, indent=2, ensure_ascii=False), encoding="utf-8")

        eval_records = [{k: v for k, v in row.items() if k != "eval_episode_rows"} for row in history if row.get("did_evaluate")]
        if eval_records:
            pd.DataFrame(eval_records).to_csv(EVAL_HISTORY_CSV, index=False)

        print({
            "chunk": chunk_idx,
            "env_steps": current_env_steps,
            "actor_loss": round(record["actor_loss"], 4),
            "actor_total_loss": round(record["actor_total_loss"], 4),
            "critic_loss": round(record["critic_loss"], 4),
            "entropy": round(record["entropy"], 4),
            "mean_delta": round(record["mean_delta"], 4),
            "baseline_mean": round(record["baseline_mean"], 4),
            "q_mean": round(record["q_mean"], 4),
            "did_evaluate": record["did_evaluate"],
            "eval_reward": None if not record["did_evaluate"] else round(record["eval_episode_reward"], 4),
            "balanced_score": None if not record["did_evaluate"] else round(record["balanced_score"], 4),
        })
finally:
    agent.close()


In [ ]:
history_df = pd.read_csv(TRAIN_HISTORY_CSV)
history_df.tail()

In [ ]:

history_df = pd.read_csv(TRAIN_HISTORY_CSV)
eval_df = pd.read_csv(EVAL_HISTORY_CSV) if EVAL_HISTORY_CSV.exists() else pd.DataFrame()

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.ravel()

train_plots = [
    ("actor_loss", "Actor PG Loss"),
    ("actor_total_loss", "Actor Total Loss"),
    ("critic_loss", "Critic Loss"),
    ("entropy", "Entropy"),
    ("mean_delta", "Mean Delta"),
    ("baseline_mean", "Baseline Mean"),
]
for idx, (col, title) in enumerate(train_plots):
    axes[idx].plot(history_df["env_steps"], history_df[col], marker="o")
    axes[idx].set_title(title)
    axes[idx].set_xlabel("Env Steps")
    axes[idx].grid(alpha=0.3)

if not eval_df.empty:
    axes[6].plot(eval_df["env_steps"], eval_df["eval_episode_reward"], marker="o", label="Reward")
    axes[6].plot(eval_df["env_steps"], eval_df["balanced_score"], marker="s", label="Balanced")
    axes[6].set_title("Reward vs Balanced Score")
    axes[6].set_xlabel("Env Steps")
    axes[6].legend()
    axes[6].grid(alpha=0.3)

    axes[7].plot(eval_df["env_steps"], eval_df["eval_coverage_rate"], marker="o", label="Coverage")
    axes[7].plot(eval_df["env_steps"], eval_df["eval_transport_rate"], marker="s", label="Transport")
    axes[7].set_title("Coverage and Transport")
    axes[7].set_xlabel("Env Steps")
    axes[7].legend()
    axes[7].grid(alpha=0.3)

    axes[8].plot(eval_df["env_steps"], eval_df["eval_delivered_cargoes"], marker="d")
    axes[8].set_title("Delivered Cargoes")
    axes[8].set_xlabel("Env Steps")
    axes[8].grid(alpha=0.3)
else:
    for idx in [6, 7, 8]:
        axes[idx].axis('off')

plt.tight_layout()
plt.show()


In [ ]:

history_df = pd.read_csv(TRAIN_HISTORY_CSV)
eval_df = pd.read_csv(EVAL_HISTORY_CSV) if EVAL_HISTORY_CSV.exists() else pd.DataFrame()

print("T?m t?t train:")
last_train = history_df.iloc[-1]
print(f"- Actor PG loss cu?i: {last_train['actor_loss']:.4f}")
print(f"- Actor total loss cu?i: {last_train['actor_total_loss']:.4f}")
print(f"- Critic loss cu?i: {last_train['critic_loss']:.4f}")
print(f"- Entropy cu?i: {last_train['entropy']:.4f}")
print(f"- Baseline mean cu?i: {last_train['baseline_mean']:.4f}")
print(f"- Q mean cu?i: {last_train['q_mean']:.4f}")

if not eval_df.empty:
    best_reward = eval_df.sort_values('eval_episode_reward', ascending=False).iloc[0]
    best_balanced = eval_df.sort_values('balanced_score', ascending=False).iloc[0]
    print()
    print("T?m t?t eval:")
    print(f"- Best reward: {best_reward['eval_episode_reward']:.3f} t?i {int(best_reward['env_steps'])} env steps")
    print(f"- Best balanced score: {best_balanced['balanced_score']:.3f} t?i {int(best_balanced['env_steps'])} env steps")
    print(f"- Coverage/Transport t?i best balanced: {best_balanced['eval_coverage_rate']:.3f} / {best_balanced['eval_transport_rate']:.3f}")
